In [0]:
%pip install openai sentence-transformers "chromadb==0.5.23" "numpy<2.0"
dbutils.library.restartPython()

In [0]:
%pip install -U mlflow
dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
def restore_from_dbfs():
    """Restore chroma store from /dbfs/ backup"""
    backup = "/dbfs/portfolio_assistant/chroma_backup.zip"
    if os.path.exists(backup):
        os.makedirs(CHROMA_PATH, exist_ok=True)
        shutil.unpack_archive(backup, CHROMA_PATH)
        print("✓ Restored from /dbfs/")
        return True
    print("No backup found — need to re-ingest")
    return False

In [0]:
import os
import hashlib
import re
from pathlib import Path
from typing import List, Dict

from sentence_transformers import SentenceTransformer
import chromadb
from openai import OpenAI

print("Imports OK")

In [0]:
# Same path as before — reconnects to the data you already ingested
CHROMA_PATH = "/tmp/portfolio_assistant/chroma_store"

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

restore_from_dbfs()
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

print(f"Connected — {collection.count()} chunks in store")

In [0]:
def retrieve(question: str, n_results: int = 3) -> List[Dict]:
    """
    Embed the question, find the most similar chunks.
    Returns a list of dicts with text and metadata.
    """
    query_vector = model.encode([question]).tolist()

    results = collection.query(
        query_embeddings=query_vector,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append({
            "text":       doc,
            "source":     meta["source"],
            "section":    meta["section"],
            "similarity": round((1 - dist) * 100, 1)
        })

    return chunks

In [0]:
def build_prompt(question: str, chunks: List[Dict]) -> str:
    """
    Combine retrieved chunks + question into a prompt for the LLM.

    Why a system/user split?
    OpenAI's API takes two message types:
    - system: sets the LLM's behaviour and role (read once, sets context)
    - user: the actual question

    The context goes in the system message so the LLM treats it as
    ground truth, not as part of the conversation.
    """
    context_blocks = []
    for i, chunk in enumerate(chunks):
        context_blocks.append(
            f"--- source {i+1}: {chunk['source']} / {chunk['section']} ---\n"
            f"{chunk['text']}"
        )

    context = "\n\n".join(context_blocks)

    system_prompt = f"""You are a helpful assistant for Aria Mostajeran's portfolio website.
Aria is an MSc Data Science & AI student at TU/e (Eindhoven) finishing in 2025,
actively looking for ML/data science roles.

Answer questions about Aria's skills, projects, experience, and background
using ONLY the context provided below. Be specific and concrete — use actual
names, numbers, and technologies from the context. Keep answers concise (2-4
sentences unless a longer answer is clearly needed).

If the answer is not in the context, say:
"I don't have that information in my knowledge base."
Do NOT make up or infer anything not explicitly stated in the context.

CONTEXT:
{context}"""

    return system_prompt

In [0]:
# Set your API key
# Set your OpenAI API key in .env file or environment
# os.environ["OPENAI_API_KEY"] = "your-key-here"

client = OpenAI()

def ask(question: str, verbose: bool = False) -> str:
    """
    Full RAG pipeline in one function:
    1. Retrieve relevant chunks
    2. Build prompt
    3. Call LLM
    4. Return answer

    verbose=True shows which chunks were used — useful for debugging
    """
    chunks = retrieve(question, n_results=3)

    if verbose:
        print("Retrieved chunks:")
        for c in chunks:
            print(f"  {c['similarity']}% — {c['source']} / {c['section']}")
        print()

    system_prompt = build_prompt(question, chunks)

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",      # cheap, fast, good enough for this
        messages=[
            {"role": "system",  "content": system_prompt},
            {"role": "user",    "content": question}
        ],
        temperature=0.2,            # low temperature = more factual, less creative
        max_tokens=300
    )

    return response.choices[0].message.content

In [0]:
# Cell — re-ingest both files
from pathlib import Path
import re, hashlib

SKIP_SECTIONS = {"links", "contact", "languages", "interests"}

def chunk_markdown(text: str, source: str) -> List[Dict]:
    chunks = []
    sections = re.split(r'\n(?=## )', text.strip())
    for section in sections:
        if not section.strip():
            continue
        lines = section.strip().split('\n')
        heading = lines[0].replace('#', '').strip() if lines[0].startswith('#') else "intro"
        content = '\n'.join(lines).strip()
        if len(content) < 150:
            continue
        if heading.lower() in SKIP_SECTIONS:
            continue
        chunk_id = hashlib.md5(f"{source}::{heading}".encode()).hexdigest()
        chunks.append({
            "id": chunk_id, "text": content,
            "source": source, "section": heading,
            "type": "project" if "projects/" in source else "cv"
        })
    return chunks

def embed_and_store(chunks):
    if not chunks:
        return
    texts = [c["text"] for c in chunks]
    vectors = model.encode(texts, convert_to_numpy=True)
    collection.upsert(
        ids        = [c["id"]    for c in chunks],
        embeddings = [v.tolist() for v in vectors],
        documents  = [c["text"]  for c in chunks],
        metadatas  = [{"source": c["source"], "section": c["section"], "type": c["type"]} for c in chunks]
    )

def ingest_file(filepath: str):
    filepath = Path(filepath)
    text   = filepath.read_text(encoding="utf-8")
    source = filepath.name
    chunks = chunk_markdown(text, source)
    embed_and_store(chunks)
    print(f"✓ {filepath.name} — {len(chunks)} chunks")

ingest_file("/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/knowledge/projects/portfolio_site.md")
ingest_file("/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/knowledge/cv.md")
print(f"\nTotal chunks: {collection.count()}")

In [0]:
questions = [
    "Does Aria have experience with Docker?",
    "What is Aria's strongest ML project?",
    "What tech stack does Aria use for his portfolio website?",
    "Has Aria worked with transformers?",
    "What makes Aria's portfolio different from a normal CV?",
    "Is Aria available for work?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print()

In [0]:
print(ask("What is Aria's salary expectation?"))
print(ask("What did Aria do last weekend?"))